In [ ]:
#========================================
# Import required packages
#========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle, Ellipse
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import json
import sys
sys.path.append(r"C:\Users\F.Turner\Documents\00. Analyses")
import use_funcs
from use_funcs import find_project_root

# **Create Helper Functions**

Define plotting palette and project path helpers used throughout the notebook.

In [ ]:
#========================================
# Define Save the Children colour palette
#========================================

sc_colors = {'red': '#da291c',
             'medium red' : '#ed7b73',
             'light red': '#f9d3d0',
             'purple': '#ae90c3',
             'medium purple': '#cebcdb',
             'light purple': '#efe9f3',
             'yellow': '#fecf28',
             'medium yellow': '#ffeca9',
             'light yellow': '#fff5d4',
             'blue': '#99cccc',
             'medium blue' : '#d6ebeb',
             'light blue' : '#ebf5f5',
             'green' : '#45b283',
             'medium green' : '#8dd3b5',
             'light green' : '#d9f0e6',
             'grey' : '#e7e6e6'}

mpl.rcParams['font.family'] = 'Calibri' 

In [ ]:
#========================================
# Configure project paths and output folders
#========================================

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "path_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PATHS = json.load(f)

IMP_DIR = (PROJECT_ROOT / PATHS["imports_dir"]).resolve()
HO_DIR = (PROJECT_ROOT / PATHS["handoff_dir"]).resolve()
EXP_DIR = (PROJECT_ROOT / PATHS["exports_dir"]).resolve()
FIG_DIR = (PROJECT_ROOT / PATHS["figures_dir"]).resolve()
TAB_DIR = (PROJECT_ROOT / PATHS["tables_dir"]).resolve()

for folder in [IMP_DIR, HO_DIR, EXP_DIR, FIG_DIR, TAB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMP_DIR:", IMP_DIR)
print("HO_DIR:", HO_DIR)
print("EXP_DIR:", EXP_DIR)
print("FIG_DIR:", FIG_DIR)
print("TAB_DIR:", TAB_DIR)

PROJECT_ROOT: C:\Users\F.Turner\Documents\00. Analyses\Education Financing
IMP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Data
HO_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Handoff
EXP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results
FIG_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\figures
TAB_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\tables


# **Load Data**

Load the prepared merged datasets used in the modeling and visualization steps.

In [ ]:
#========================================
# Load core analysis datasets
#========================================

all_data = pd.read_csv(IMP_DIR / "all_data.csv")
lays_panels = pd.read_csv(IMP_DIR / "lays_panels.csv")

In [ ]:

#=====================================================================
# Prepare panel dataset for plotting change in education expenditure vs change in LAYS
#=====================================================================

df = lays_panels.copy()
df = lays_panels[lays_panels['panel'].isin([1, 2, 3, 4, 5])]

agg_map = {col: 'mean' for col in df.columns if col not in ['iso3', 'income_group']}
agg_map['income_group'] = 'first'
df = df.groupby('iso3', as_index=False).agg(agg_map)
# df = df[df['income_group'].isin(['Low income, Low middle income', 'Upper middle income', 'High income'])]

#=============================================================================================
# Plot change in education budget share against change in LAYS
#=============================================================================================


fig, ax = plt.subplots(2, 1, figsize=(10, 16))

ax[0].add_patch(Rectangle(( -0.03, -0.4), 0.06, 0.4, fill=True, color=sc_colors['light red'], alpha=1))
ax[0].add_patch(Rectangle((-0.03, 0), 0.06, 0.4, fill=True, color=sc_colors['light green'], alpha=1))


ax[0].axhline(0, color='grey', linestyle='--')
ax[0].axvline(0, color='grey', linestyle='--')

sns.scatterplot(data=df, x='delta_expenditure_pctbudget_uis', 
                y='delta_lays',
                hue='income_group', 
                # style='panel',
                palette={"Low income": sc_colors['green'], 
                           "Lower middle income": sc_colors['red'], 
                           "Upper middle income": sc_colors['purple'], 
                           "High income": sc_colors['blue'],
                            'Not classified': sc_colors['grey']},
                s=80,
                edgecolor='black',
                alpha=0.7,
                ax=ax[0]) 


ax[0].set_xlim(-0.026, 0.025)
ax[0].set_xlabel("Change in Education Expenditure as % of Budget")
ax[0].set_ylabel("Annualised Change in LAYS")
ax[0].set_title("Annualised Change in Education Expenditure as \n% of Budget vs Annualised Change in LAYS", fontsize=16, fontname='Oswald')
ax[0].set_ylim(-.4, .4)

ax[0].grid(True, linestyle='--', alpha=0.5)
ax[0].set_facecolor('whitesmoke')
ax[0].spines[['top', 'right']].set_visible(False)

ax[0].legend(title='Income Group', bbox_to_anchor=(1.01, 1), loc='upper left')

#=============================================================================
# Plot change in share of GDP spent on education against change in LAYS
#=============================================================================



ax[1].add_patch(Rectangle(( -0.03, -0.4), 0.06, 0.4, fill=True, color=sc_colors['light red'], alpha=1))
ax[1].add_patch(Rectangle((-0.03, 0), 0.06, 0.4, fill=True, color=sc_colors['light green'], alpha=1))


ax[1].axhline(0, color='grey', linestyle='--')
ax[1].axvline(0, color='grey', linestyle='--')

sns.scatterplot(data=df, x='delta_expenditure_pctgdp', 
                y='delta_lays',
                hue='income_group', 
                # style='panel',
                palette={"Low income": sc_colors['green'], 
                           "Lower middle income": sc_colors['red'], 
                           "Upper middle income": sc_colors['purple'], 
                           "High income": sc_colors['blue'],
                            'Not classified': sc_colors['grey']},
                s=80,
                edgecolor='black',
                alpha=0.7,
                ax=ax[1]) 


ax[1].set_xlim(-0.026, 0.025)
ax[1].set_xlabel("Change in Education Expenditure as % of Budget")
ax[1].set_ylabel("Annualised Change in LAYS")
ax[1].set_title("Annualised Change in Education Expenditure as \n% of Budget vs Annualised Change in LAYS", fontsize=16, fontname='Oswald')
ax[1].set_ylim(-.4, .4)

ax[1].grid(True, linestyle='--', alpha=0.5)
ax[1].set_facecolor('whitesmoke')
ax[1].spines[['top', 'right']].set_visible(False)

ax[1].legend(title='Income Group', bbox_to_anchor=(1.01, 1), loc='upper left')


fig.savefig(FIG_DIR / "delta_expenditure_vs_delta_lays.png", dpi=300, bbox_inches="tight")